In [1]:
import numpy as np
np.random.seed(42)

from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance

In [3]:
from indices import *

import tracemalloc
import time

In [4]:
data = [None]*2
data[0] = np.load("dataset_cropped_64_npy\\TRAIN_HEALTHY_even.npy") # healthy
data[1] = np.load("dataset_cropped_64_npy\\TRAIN_STRESSED_even.npy") # stressed

In [5]:
labels = np.concatenate([np.zeros(data[0].shape[1]), np.ones(data[1].shape[1])])
labels.shape

(38978,)

In [9]:
data[1].shape

(12, 19489)

In [6]:
tracemalloc.start()

In [7]:
band_indexes = list(range(1, 12))
encoder = IndicesClassEncoderEq([HueSimp], band_indexes)

feature_id = []
features = []
for i in range(encoder.total_length):
    index = encoder.getIndex(i)
    a = index.args
    if a[1] <= a[2]:
        continue

    feature_id.append(i)
    features.append(np.concatenate([index.getValue(data[0]), index.getValue(data[1])]))

features = np.array(features).swapaxes(0, 1)

In [8]:
features.shape

(38978, 605)

In [ ]:
time_start = time.time()

dt = RandomForestClassifier(random_state=42)
dt.fit(features, labels)

result = permutation_importance(
    dt, features, labels, random_state=42, n_jobs=8
)



In [10]:
selected = np.array(result.importances_mean).argsort()[::-1][:3]

In [11]:
time_end = time.time()
print("Time:", time_end - time_start, "sec")
print("MEM usage:", np.array(tracemalloc.get_traced_memory()) / 1024**2, "mb")
tracemalloc.stop()

Time: 2584.0266473293304 sec
MEM usage: [180.57380962 359.97837543] mb


In [12]:
mapping = {
    0: "B1",
    1: "B2",
    2: "B3",
    3: "B4",
    4: "B5",
    5: "B6",
    6: "B7",
    7: "B8",
    8: "B8A",
    9: "B9",
    10: "B11",
    11: "B12"
}

for id in selected:
    index_id = feature_id[id]
    index = encoder.getIndex(index_id)
    name = getIndexName(index, mapping)
    print("Id:", index_id, "Name:", name)

Id: 13 Name: HueSimp(B4, B3, B2)
Id: 147 Name: HueSimp(B6, B4, B3)
Id: 603 Name: HueSimp(B11, B12, B6)
